In [ ]:
from datetime import datetime, timedelta, timezone
import pandas as pd
from wrappers.tpqoa import OandaClient

client = OandaClient.from_creds()

end = datetime.now(timezone.utc)

start_daily = end - timedelta(days = 365)

start_intra = end - timedelta(minutes = 360)

candles_daily = client.get_candles(
    "EUR_USD",
    start = start_daily,
    end = end,
    granularity = "D",
    price = "M",
)

candles_intra = client.get_candles(
    "EUR_USD",
    start = start_intra,
    end = end,
    granularity = "S5",
    price = "M",
)

In [ ]:
time_str, bid, ask = client.get_prices("EUR_USD")
spread = ask - bid

closes = candles_intra["c"].astype(float)
rets = closes.pct_change().dropna()


avg_spread = pd.Series(spread).mean()
intra_vol = rets.std(ddof = 1)

print(avg_spread, intra_vol)

In [ ]:
summary = client.get_account_summary()

account_balance = summary["balance"]

time_str, bid, ask = client.get_prices("EUR_USD")

mid = (bid + ask) / 2

risk_per_trade = 0.01

stop_limit = 0.002
# how much loss can be accept
stop_distance = mid * stop_limit

risk_amount = account_balance * risk_per_trade
units = risk_amount / stop_distance

# the price to exit
stop_loss = mid *(1- stop_limit)

In [ ]:
import datetime
from wrappers.tpqoa import OandaClient

# get account information 
summary = client.get_account_summary()
account_balance = summary["balance"]
open_position = summary["openTradeCount"]

print(f"[{datetime.datetime.now()}] PRE-TRADE balance: {summary['balance']}  open_position: {open_position}")

# get price
time_str, bid, ask = client.get_prices("EUR_USD")
mid = (bid+ask)/2

#get an order
order = client.place_market_order(
    instrument = "EUR_USD",
    units = 1000,
    stop_loss = mid*(1 - 0.002),
    take_profit = mid * (1+0.004),
    trailing_stop_distance = 0.0010,
)
print(f"[{datetime.datetime.now()}] ORDER SUBMITTED  trade_id: {order['trade_id']} price: {order['price']}")

# close 
client.close_trades(instrument = "EUR_USD")

In [ ]:
local_positions = {"EUR_USD": 1000}

summary = client.get_account_summary()
broker_open_count = summary["openTradeCount"]


if len(local_positions) == broker_open_count:
    print(f"positions match: open amount: {sum(local_positions.values())}")
else:
    print("number mismatch: check!")

In [ ]:
import pandas as pd
from datetime import datetime, timedelta, timezone

class BrokerAPI:

    def get_account_summary(self) -> dict[str, Any]:
        """Return balance, open trade count, account ID."""
        summary = self.client.get_account_summary()
        return {
            "accountId": summary["accountId"],
            "balance": float(summary["balance"]),
            "openTradeCount": summary["openTradeCount"],
        }

    def get_prices(self, instrument: str) -> tuple[str, float, float]:
        """Return (timestamp, bid, ask) for an instrument."""
        time_str, bid, ask = self.client.get_prices(instrument)
        mid = (bid + ask) / 2.0
        return time_str, bid, ask, mid

    def get_candles(
        self,
        instrument: str,
        *,
        granularity: str = "D",
        start: datetime | str | None = None,
        end: datetime | str | None = None,
        num_points: int | None = None,
    ) -> pd.DataFrame:
        """Fetch candles for an instrument, either by range or by count."""
        end = datetime.now(timezone.utc)
        if start is None:
            start = end - timedelta(days=365)
        candles = self.client.get_candles(
            instrument,
            granularity=granularity,
            start=start,
            end=end,
        )
        closes = candles["mid_close"].astype(float)
        return candles

    def market_order(
        self,
        instrument: str,
        *,
        direction: str,
        size: float,
        stop_distance: float | None = None,
        limit_distance: float | None = None,
        guaranteed_stop: bool = False,
        force_open: bool = True,
        time_in_force: str | None = None,
    ) -> dict[str, Any]:
        """Place a simple market order"""
        order = self.client.market_order(
            instrument,
            direction=direction,
            size=size,
            stop_distance=stop_distance,
            limit_distance=limit_distance,
        )
        return order

    def close_position(
        self,
        deal_id: str,
        *,
        instrument: str,
        direction: str,
        size: float,
        time_in_force: str | None = None,
    ) -> dict[str, Any]:
        """Close existing position"""
        close = self.client.close_position(
            deal_id=deal_id,
            instrument=instrument,
            direction=direction,
            size=size,
        )
        return close

In [ ]:
 def _format_prices(self, prices: list[dict[str, Any]], version: str) -> pd.DataFrame:
        if isinstance(prices, pd.DataFrame):
            return prices
        rows = []
        for entry in prices:
            timestamp = self._parse_datetime(entry)
            row: dict[str, Any] = {"DateTime": timestamp}
            for field in ("openPrice", "highPrice", "lowPrice", "closePrice"):
                metric = field.replace("Price", "").lower()
                data = entry.get(field, {})
                bid = self._to_float(data.get("bid"))
                ask = self._to_float(data.get("ask"))
                row[f"bid_{metric}"] = bid
                row[f"ask_{metric}"] = ask
                if bid is not None and ask is not None:
                    row[f"mid_{metric}"] = (bid + ask) / 2
                else:
                    row[f"mid_{metric}"] = None
            row["volume"] = self._to_float(entry.get("lastTradedVolume"))
            rows.append(row)
        frame = pd.DataFrame(rows)
        frame = frame.set_index("DateTime")
        frame.index = pd.to_datetime(frame.index)
        frame.sort_index(inplace=True)
        return frame

In [ ]:
## I am not sure what this question is asking about. 
## it seems to me like I set up some number and then print it out? but that doesn't make sense

In [ ]:
def _ensure_session(self) -> None:
        if not self._session_active:
            self.service.create_session()
            self._session_active = True
            
def close_session(self) -> None:
        if self._session_active:
            try:
                self.service.logout()
            except AttributeError:
                LOG.debug("IG logout endpoint not available in stubbed service.")
            self._session_active = False
            
## and in the folloinwg code for every function, self._ensure_session()

In [ ]:
from datetime import datetime, timedelta, timezone
from wrappers.tpqoa import OandaClient
from wrappers.tpqig import IGClient

end = datetime.now(timezone.utc)
start = end - timedelta(days=365)

# fetch data from both brokers
oanda_client = OandaClient.from_creds()
ig_client = IGClient.from_creds()

oanda_candles = oanda_client.get_candles("EUR_USD", granularity="D", start=start, end=end, price="M")

ig_candles = ig_client.get_candles("EUR/USD", granularity="D", start=start, end=end)

# map to common schema
oanda_close = oanda_candles["c"].astype(float)
ig_close = ig_candles["mid_close"].astype(float)

# compute returns
oanda_rets = oanda_close.pct_change().dropna()
ig_rets = ig_close.pct_change().dropna()

# summarise differences
print("Oanda")
print(f"obs: {len(oanda_rets)}")
print(f"mean: {oanda_rets.mean():.4f}")
print(f"vol:  {oanda_rets.std(ddof=1):.4f}")

print("IG")
print(f"obs: {len(ig_rets)}")
print(f"mean: {ig_rets.mean():.4f}")
print(f"vol:  {ig_rets.std(ddof=1):.4f}")

In [ ]:
def daily_report(
    equity: pd.Series,
    trades: pd.DataFrame,
    name: str = "daily_report",
) -> Path:    
    table = equity_table(equity, vol_window=20)
    rets = table["return"].dropna()

    summary = pd.DataFrame({
        "daily_return":    [rets.iloc[-1]],
        "mean_return":     [rets.mean()],
        "volatility":      [rets.std(ddof=1)],
        "ann_vol":         [table["ann_vol"].iloc[-1]],
        "max_drawdown":    [table["drawdown"].min()],
        "current_drawdown":[table["drawdown"].iloc[-1]],
        "trade_count":     [len(trades)],
    })
    print(summary.to_string(index=False))

    cfg = EquityReportConfig(name=name)
    return save_equity_report(equity, cfg)

In [ ]:
import pandas as pd
from ch16_reporting_monitoring import build_demo_equity

equity = build_demo_equity()

# sort the table for later identify the position and  compute drawdown manually
equity = equity.sort_index()
running_max = equity.cummax()
drawdown = equity / running_max - 1.0

# find date of max drawdown
max_dd_val = drawdown.min()
max_dd_date = drawdown.index[drawdown == max_dd_val][0]

#  after the drawdown date
recovery = drawdown[drawdown.index >= max_dd_date]

#first date where drawdown returns to 0
recovery_date = recovery.index[recovery >= 0][0]

# summary
print(f"max drawdown:    {max_dd_val:.4f}")
print(f"drawdown date:   {max_dd_date.date()}")
print(f"recovery date:   {recovery_date.date()}")
print(f"duration (days): {(recovery_date - max_dd_date).days}")

## I got a feeling that this logic might not work. 

In [ ]:
def summarise_trades(trades: pd.DataFrame) -> pd.DataFrame:
    
    required = {"instrument", "signal", "return"}
    missing = required.difference(trades.columns)
    if missing:
        raise ValueError(f"missing columns in trades: {missing}")
    
    grouped = (
        trades.groupby(["signal", "instrument"], as_index=False)
        .agg(
            mean_return=("return", "mean"),
            total_return=("return", "sum"),
            trade_count=("return", "count"),
        )
        .sort_values(
            by=["signal", "total_return"],
            ascending=[True, False],
        )
    )
    return grouped

In [ ]:
import q
import pandas as pd
from datetime import datetime, timezone

def check_alerts(
    equity: pd.Series,
    ticks: pd.DataFrame,
    vol_window: int = 20,
    vol_threshold: float = 0.30,
    drawdown_limit: float = -0.10,
    max_gap_seconds: int = 300,
) -> None:

    rets = equity.pct_change().dropna()

    # unusual volatility check
    rolling_vol = rets.rolling(vol_window).std(ddof=1) * (252 ** 0.5)
    current_vol = rolling_vol.iloc[-1]
    if current_vol > vol_threshold:
        q(f"ALERT: unusual volatility {current_vol:.4f}")

    # sudden drawdown check
    running_max = equity.cummax()
    drawdown = equity / running_max - 1.0
    current_dd = drawdown.iloc[-1]
    if current_dd < drawdown_limit:
        q(f"ALERT: drawdown {current_dd:.4f}")

    #  data outage check
   last_tick = ticks.index[-1]

# if last_tick is timezone-naive, localize it to UTC
   if last_tick.tzinfo is None:
        last_tick = last_tick.tz_localize("UTC")
   else:
        last_tick = last_tick.tz_convert("UTC")  # convert to UTC if different tz

   now = pd.Timestamp.now(tz="UTC")
   gap_seconds = (now - last_tick).seconds
   if gap_seconds > max_gap_seconds:
       q(f"ALERT: no ticks for {gap_seconds}s")


In [ ]:
## are we requested to write the code to generate several chart there or just describe?

##Dashboard Layout

##Performance:

# Equity curve (normalised wealth index)
# Cumulative and period returns
# can also based on the security / security group/ sectors
# can be obsolute value too

## Risk:

# Drawdown chart
# Current and max drawdown values
# Rolling annualised volatility

## Operational:

# Trade count
# Last tick timestamp (data outage check)
# Any active alerts (price trigger, volatility trigger, etc)